# Self-Attention Generative Adversarial Network (SAGAN) Trained on CelebA Dataset

Version 2

Source:

https://github.com/heykeetae/Self-Attention-GAN

Adapted by:

Antonio Esteves @ UMinho, Jul 2024

In [ ]:
import cv2
import numpy as np
import os
import shutil
import datetime
import random
import sys
import time
import yaml
import wandb
import PIL.Image             as     Image
from   natsort               import natsorted

import torch
import torchvision.datasets  as     dset
from   torchvision           import transforms
import torch.nn              as     nn
from   torch.nn              import Parameter
import torchvision.utils     as     vutils
from   torch.backends        import cudnn
import torch.nn.functional   as     F
from   torch.nn.utils        import spectral_norm
from   torch.nn.init         import xavier_uniform_
from   torch.utils.data      import DataLoader, Dataset
from   torchvision.utils     import save_image, make_grid
import torchvision.datasets  as     dsets
from   torch.autograd        import Variable
from   torchinfo             import summary

import matplotlib.pyplot     as     plt

## Read the Configuration

In [ ]:
LOAD_TRAINED_MODEL        = False
SKIP_TRAIN_MODEL          = False

CONFIG_FILE = 'config/config_sagan_celeba_128x128_07.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

config["models_path"]  = os.path.join( config["root_dir"],  config["models_dir"],  config["experiment_name"])
config["results_path"] = os.path.join( config["root_dir"],  config["results_dir"], config["experiment_name"])

In [ ]:
if (config['crop_size'] == 'None'):
    config['crop_size'] = None

In [ ]:
print('parameters:')
for key, value in config.items():
    print(f'\t{key}: {value}')

In [ ]:
# Setup device agnostic code

device       = "cuda" if torch.cuda.is_available() else "cpu"
# device = 'cpu'

print(f'Using {device} for computing')

## Utility functions

In [ ]:
def make_folder(path):
    '''
    Creates the folder 'path' if it does not exist.
    '''
    if not os.path.exists(path,):
        os.makedirs(path)


def denorm(x):
    '''
    Denormalizes the tensor 'x', by adding '1', divinding by '2',
    # and clipping the values to the range [0,1].
    '''
    out = (x + 1) / 2
    return out.clamp_(0, 1)

def time_format(seconds: int) -> str:
    '''
    Converts a time in seconds to a formated string in the form: 01D:12H:34m:56s.
    '''
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'

## Login into Weights and Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights &and
Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(project='GAN_CelebA_2', entity='ajesteves', config=config_wandb)

## Create a custom Dataset from the images in a folder

In [ ]:
class CustomDataSet(Dataset):

    def __init__(self, root_dir, transform):
        self.root_dir     = root_dir
        self.transform    = transform
        self.all_images   = os.listdir(root_dir)
        self.total_images = natsorted(self.all_images)

    def __len__(self):
        return len(self.total_images)

    def __getitem__(self, idx):
        img_loc      = os.path.join(self.root_dir, self.total_images[idx])
        image        = Image.open(img_loc).convert("RGB")
        tensor_image = self.transform(image)
        return tensor_image

## Create a DataLoader

Define the transformation that will be applied to the images:

* convert the images to tensors
* crop the images
* resize the images
* normalize the images.

Instantiate a Custom Dataset
Create a training DataLoader

In [ ]:
class Data_Loader():
    '''
    DataLoader class that works with LSUN and CelebA datasets.
    '''
    def __init__(
            self,
            dataset,
            images_path,
            image_size,
            crop_size,
            batch_size,
            shuffle = True,
        ):
        self.dataset    = dataset
        self.path       = images_path
        self.image_size = image_size
        self.crop_size  = crop_size
        self.batch_size = batch_size
        self.shuffle    = shuffle

    def transform(self, resize, totensor, normalize, centercrop):

        options = []

        if totensor:
            options.append(transforms.ToTensor())

        if centercrop:
            offset_height = (218 - self.crop_size) // 2
            offset_width  = (178 - self.crop_size) // 2
            crop = lambda x: x[:, offset_height:offset_height + self.crop_size, offset_width:offset_width + self.crop_size]
            options.append(transforms.Lambda(crop))

        if resize:
            options.append(transforms.Resize((self.image_size, self.image_size)))

        if normalize:
            options.append(transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)))
        transform = transforms.Compose(options)

        return transform

    def load_lsun(self, classes='church_outdoor_train'):
        transforms = self.transform(True, True, True, False)
        dataset    = dsets.LSUN(self.path, classes=[classes], transform=transforms)
        return dataset

    def load_celeb(self):
        transforms = self.transform(True, True, True, True)
        dataset = CustomDataSet(
            root_dir  = self.path,
            transform = transforms,
        )
        return dataset


    def loader(self):
        if self.dataset == 'lsun':
            dataset = self.load_lsun()
        elif self.dataset == 'celeb':
            dataset = self.load_celeb()

        loader = torch.utils.data.DataLoader(
            dataset     = dataset,
            batch_size  = self.batch_size,
            shuffle     = self.shuffle,
            num_workers = 2,
            drop_last   = True,
        )
        return loader

## Let us check if everything works fine and display a few real images.

In [ ]:
def check_dataloader(dataset, images_path, image_size, crop_size, batch_size, shuffle=True):
    NR, NC    = 3, 3

    DLoader = Data_Loader(
        dataset     = dataset,
        images_path = images_path,
        image_size  = image_size,
        crop_size   = crop_size,
        batch_size  = batch_size,
        shuffle     = shuffle,
    )

    loader = DLoader.loader()
    imgs   = next(iter(loader))

    print(f'Batch of images shape: {imgs.shape}')   # BS, Ch, H, W

    if NR*NC > imgs.shape[0]:
        NR = 2
        if NR*NC > imgs.shape[0]:
            NR = 1
            if NR*NC > imgs.shape[0]:
                NC = 2

    _, ax    = plt.subplots(NR, NC, figsize=(3*NC,3*NR))
    plt.suptitle(
        f'Some real images of {dataset} dataset',
        fontsize   = 15,
        fontweight = 'bold',
    )

    index = 0
    for r in range(NR):
        for c in range(NC):
            index += 1
            if NR==1:
                ax[c].imshow((imgs[index].permute(1,2,0)+1)/2) 
            else:
                ax[r][c].imshow((imgs[index].permute(1,2,0)+1)/2) 

In [ ]:
# Check the dataloader

check_dataloader(
    dataset     = config["dataset"],
    images_path = config["data_path"],
    image_size  = config["image_size"],
    crop_size   = config["crop_size"],
    batch_size  = config["batch_size"],
    shuffle     = True,
)

## Spectral Normalization Layer

In [ ]:
def l2normalize(v, eps=1e-12):
    return v / (v.norm() + eps)


class SpectralNorm(nn.Module):
    '''
    Spectral normalization class.
    '''
    def __init__(self, module, name='weight', power_iterations=1):
        super(SpectralNorm, self).__init__()
        self.module = module
        self.name = name
        self.power_iterations = power_iterations
        if not self._made_params():
            self._make_params()

    def _update_u_v(self):
        u = getattr(self.module, self.name + "_u")
        v = getattr(self.module, self.name + "_v")
        w = getattr(self.module, self.name + "_bar")

        height = w.data.shape[0]
        for _ in range(self.power_iterations):
            v.data = l2normalize(torch.mv(torch.t(w.view(height,-1).data), u.data))
            u.data = l2normalize(torch.mv(w.view(height,-1).data, v.data))

        # sigma = torch.dot(u.data, torch.mv(w.view(height,-1).data, v.data))
        sigma = u.dot(w.view(height, -1).mv(v))
        setattr(self.module, self.name, w / sigma.expand_as(w))

    def _made_params(self):
        try:
            u = getattr(self.module, self.name + "_u")
            v = getattr(self.module, self.name + "_v")
            w = getattr(self.module, self.name + "_bar")
            return True
        except AttributeError:
            return False


    def _make_params(self):
        w = getattr(self.module, self.name)

        height = w.data.shape[0]
        width = w.view(height, -1).data.shape[1]

        u = Parameter(w.data.new(height).normal_(0, 1), requires_grad=False)
        v = Parameter(w.data.new(width).normal_(0, 1), requires_grad=False)
        u.data = l2normalize(u.data)
        v.data = l2normalize(v.data)
        w_bar = Parameter(w.data)

        del self.module._parameters[self.name]

        self.module.register_parameter(self.name + "_u", u)
        self.module.register_parameter(self.name + "_v", v)
        self.module.register_parameter(self.name + "_bar", w_bar)


    def forward(self, *args):
        self._update_u_v()
        return self.module.forward(*args)

## Self-attention Layer

In [ ]:
class Self_Attn(nn.Module):
    """
    Self-attention Layer.
    """
    def __init__(self, in_dim, activation):

        super(Self_Attn, self).__init__()

        self.chanel_in  = in_dim
        self.activation = activation
        
        # Build the module
        self.query_conv = nn.Conv2d( # 1 X 1 convolution
            in_channels  = in_dim,
            out_channels = in_dim // 2,
            kernel_size  = 1,
        )
        self.key_conv   = nn.Conv2d( # 1 X 1 convolution
            in_channels  = in_dim,
            out_channels = in_dim // 2,
            kernel_size  = 1,
        )
        self.value_conv = nn.Conv2d( # 1 X 1 convolution
            in_channels  = in_dim,
            out_channels = in_dim,
            kernel_size  = 1,
        )
        self.gamma      = nn.Parameter(torch.zeros(1))
        self.softmax    = nn.Softmax(dim = -1)

    def forward(self, x):
        """
        inputs:
            x : input feature maps( BS * C * H * W)
        returns:
            out       -> self-attention value added to the input feature 'x'
            attention -> BS * N * N (N is Width*Height)
        """
        bs, C, height, width = x.size()

        # Query^T shape: [BS, H*W, C/2]
        proj_query = self.query_conv(x).view(bs, -1, height*width).permute(0,2,1)
        # Key shape:     [BS, C/2, H*W]
        proj_key   = self.key_conv(x).view(bs, -1, height*width)
        # S (att scores) shape: [BS, H*W, H*W]
        s_scores   = torch.bmm(proj_query, proj_key)  # batch matrix-matrix product

        # Attention shape: [BS, H*W, H*W]
        attention  = self.softmax(s_scores) 
        # Value shape: [BS, C, H*W]
        proj_value = self.value_conv(x).view(bs, -1, height*width)

        # 'o' shape: [BS, C, H*W]
        o          = torch.bmm(proj_value, attention.permute(0,2,1)) # batch matrix-matrix product
        # 'o' shape: [BS, C6, H, W]
        o          = o.view(bs,C,width,height)

        # 'y' shape: [BS, C6, H, W]
        y          = self.gamma * o + x

        return y, attention

## Generator Model

In [ ]:
class Generator(nn.Module):
    """
    The generator model.
    """

    def __init__(self, batch_size, image_size=64, channels=3, z_dim=128, conv_dim=64):
        super(Generator, self).__init__()
        self.image_size = image_size
        self.channels   = channels
        layer1      = []
        layer2      = []
        layer3      = []
        layer4      = []
        last        = []

        repeat_num = int(np.log2(self.image_size)) - 3   # image_size=64 -> 3, image_size=128 -> 4
        mult       = 2 ** repeat_num                     # image_size=64 -> 8, image_size=128 -> 16

        # INPUT shape: BS, z_dim
        layer1.append(
            SpectralNorm(nn.ConvTranspose2d(z_dim, conv_dim * mult, 4))
        )
        layer1.append(nn.BatchNorm2d(conv_dim * mult))
        layer1.append(nn.ReLU())
        # SHAPE: BS, 64*mult, 4, 4
        #   image_size=64  -> BS,  512, 4, 4
        #   image_size=128 -> BS, 1024, 4, 4
        curr_dim = conv_dim * mult # image_size=64 -> 512, image_size=128 -> 1024

        layer2.append(
            SpectralNorm(
                nn.ConvTranspose2d(curr_dim, int(curr_dim / 2), 4, 2, 1)
            )
        )
        layer2.append(nn.BatchNorm2d(int(curr_dim / 2)))
        layer2.append(nn.ReLU())
        # SHAPE: BS, 64*mult/2, 4*2, 4*2
        #   image_size=64  -> BS, 256, 8, 8
        #   image_size=128 -> BS, 512, 8, 8
        curr_dim = int(curr_dim / 2) # image_size=64 -> 256, image_size=128 -> 512

        layer3.append(
            SpectralNorm(
                nn.ConvTranspose2d(curr_dim, int(curr_dim / 2), 4, 2, 1)
            )
        )
        layer3.append(nn.BatchNorm2d(int(curr_dim / 2)))
        layer3.append(nn.ReLU())
        # SHAPE: BS, 64*mult/(2*2), 4*(2*2), 4*(2*2)
        #   image_size=64  -> BS, 128, 16, 16
        #   image_size=128 -> BS, 256, 16, 16
        curr_dim = int(curr_dim / 2) # image_size=64 -> 128, image_size=128 -> 256

        self.attn1 = Self_Attn(curr_dim, 'relu')
        # SHAPE: BS, 64*mult/(2*2), 64*mult/(2*2)
        #   image_size=64  -> BS, 128, 128
        #   image_size=128 -> BS, 256, 256

        layer4.append(
            SpectralNorm(
                nn.ConvTranspose2d(curr_dim, int(curr_dim / 2), 4, 2, 1)
            )
        )
        layer4.append(nn.BatchNorm2d(int(curr_dim / 2)))
        layer4.append(nn.ReLU())
        # SHAPE: BS, 64*mult/(2*2*2), 4*(2*2*2), 4*(2*2*2)
        #   image_size=64  -> BS,  64, 32, 32
        #   image_size=128 -> BS, 128, 32, 32
        curr_dim = int(curr_dim / 2) # image_size=64 -> 64, image_size=128 -> 128

        self.attn2 = Self_Attn(curr_dim, 'relu')
        # SHAPE: BS, 64*mult/(2*2*2), 64*mult/(2*2*2)
        #   image_size=64  -> BS,  64,  64
        #   image_size=128 -> BS, 128, 128

        # ---------------------------------------
        if self.image_size == 128:
            layer5 = []
            layer5.append(
                SpectralNorm(
                    nn.ConvTranspose2d(curr_dim, int(curr_dim / 2), 4, 2, 1)
                )
            )
            layer5.append(nn.BatchNorm2d(int(curr_dim / 2)))
            layer5.append(nn.ReLU())
            self.l5  = nn.Sequential(*layer5)
            # SHAPE: BS, 64*mult/(2*2*2*2), 4*(2*2*2*2), 4*(2*2*2*2)
            #   image_size=128 -> BS, 64, 64, 64
            curr_dim = int(curr_dim / 2) # image_size=128 -> 64
        # ---------------------------------------

        self.l1 = nn.Sequential(*layer1)
        self.l2 = nn.Sequential(*layer2)
        self.l3 = nn.Sequential(*layer3)
        self.l4 = nn.Sequential(*layer4)

        # ---------------------------------------
        if self.image_size == 128:
            self.l5 = nn.Sequential(*layer5)
        # ---------------------------------------

        last.append(nn.ConvTranspose2d(curr_dim, channels, 4, 2, 1))
        last.append(nn.Tanh())
        self.last = nn.Sequential(*last)
        # SHAPE:
        #   image_size=64  -> BS, channels, 4*(2*2*2*2), 4*(2*2*2*2)     = BS, 3, 64, 64
        #   image_size=128 -> BS, channels, 4*(2*2*2*2*2), 4*(2*2*2*2*2) = BS, 3, 128, 128

    def forward(self, z):
        z       = z.view(z.size(0), z.size(1), 1, 1)
        out     = self.l1(z)
        out     = self.l2(out)
        out     = self.l3(out)
        out, p1 = self.attn1(out)
        out     = self.l4(out)
        out, p2 = self.attn2(out)
        # ---------------------------------------
        if self.image_size == 128:
            out = self.l5(out)
        # ---------------------------------------
        out     = self.last(out)

        return out, p1, p2

## Discriminator Model

In [ ]:
class Discriminator(nn.Module):
    """
    The discriminator model.
    """
    def __init__(self, batch_size=64, image_size=64, channels=3, conv_dim=64):
        super(Discriminator, self).__init__()
        self.image_size = image_size
        self.channels   = channels
        layer1      = []
        layer2      = []
        layer3      = []
        layer4      = []
        last        = []

        layer1.append(
            SpectralNorm(nn.Conv2d(channels, conv_dim, 4, 2, 1))
        )
        layer1.append(nn.LeakyReLU(0.1))
        # SHAPE:
        #   image_size=64  -> BS, 3, 64, 64
        #   image_size=128 -> BS, 3, 128, 128
        curr_dim = conv_dim # 64

        layer2.append(
            SpectralNorm(nn.Conv2d(curr_dim, curr_dim * 2, 4, 2, 1))
        )
        layer2.append(nn.LeakyReLU(0.1))
        # SHAPE:
        #   image_size=64  -> BS, 64*(2), 64/(2), 64/(2)   = BS, 128, 32, 32
        #   image_size=128 -> BS, 64*(2), 128/(2), 128/(2) = BS, 128, 64, 64
        curr_dim = curr_dim * 2 # 128

        layer3.append(
            SpectralNorm(nn.Conv2d(curr_dim, curr_dim * 2, 4, 2, 1))
        )
        layer3.append(nn.LeakyReLU(0.1))
        # SHAPE:
        #   image_size=64  -> BS, 64*(2*2), 64/(2*2), 64/(2*2)   = BS, 256, 16, 16
        #   image_size=128 -> BS, 64*(2*2), 128/(2*2), 128/(2*2) = BS, 256, 32, 32
        curr_dim = curr_dim * 2 # 256

        self.attn1 = Self_Attn(curr_dim, 'relu')

        layer4.append(
            SpectralNorm(nn.Conv2d(curr_dim, curr_dim * 2, 4, 2, 1))
        )
        layer4.append(nn.LeakyReLU(0.1))
        # SHAPE:
        #   image_size=64  -> BS, 64*(2*2*2), 64/(2*2*2), 64/(2*2*2)   = BS, 512,  8,  8
        #   image_size=128 -> BS, 64*(2*2*2), 128/(2*2*2), 128/(2*2*2) = BS, 512, 16, 16
        curr_dim = curr_dim*2 # 512

        self.attn2 = Self_Attn(curr_dim, 'relu')

        # ---------------------------------------
        if self.image_size == 128:
            layer5 = []
            layer5.append(
                SpectralNorm(nn.Conv2d(curr_dim, curr_dim * 2, 4, 2, 1))
            )
            layer5.append(nn.LeakyReLU(0.1))
            # SHAPE:
            #   image_size=128 -> BS, 64*(2*2*2*2), 128/(2*2*2*2), 128/(2*2*2*2) = BS, 1024, 8, 8
            curr_dim = curr_dim*2 # image_size=128 -> 1024
        # ---------------------------------------

        self.l1 = nn.Sequential(*layer1)
        self.l2 = nn.Sequential(*layer2)
        self.l3 = nn.Sequential(*layer3)
        self.l4 = nn.Sequential(*layer4)

        # ---------------------------------------
        if self.image_size == 128:
            self.l5 = nn.Sequential(*layer5)
        # ---------------------------------------

        last.append(nn.Conv2d(curr_dim, 1, 4))
        self.last = nn.Sequential(*last)
        # SHAPE: BS, 1

    def forward(self, x):
        out     = self.l1(x)
        out     = self.l2(out)
        out     = self.l3(out)
        out, p1 = self.attn1(out)
        out     = self.l4(out)
        out, p2 = self.attn2(out)

        # ---------------------------------------
        if self.image_size == 128:
            out = self.l5(out)
        # ---------------------------------------

        out     = self.last(out)

        return out.squeeze(), p1, p2

## Class to train the model

In [ ]:
class Trainer(object):
    '''
    Class to train the SAGAN model.
    '''
    def __init__(self, dataloader, config):

        # Dataloader
        self.dataloader       = dataloader

        # The name of the model and the loss to use
        self.model            = config["model"]
        self.adv_loss         = config["adv_loss"]

        # Model hyperparameters
        self.image_size       = config["image_size"]
        self.channels         = config["channels"]
        self.z_dim            = config["z_dim"]
        self.g_conv_dim       = config["g_conv_dim"]
        self.d_conv_dim       = config["d_conv_dim"]
        self.parallel         = config["parallel"]

        self.lambda_gp        = config["lambda_gp"]
        self.epochs           = config["epochs"]
        self.ncritic          = config["ncritic"]
        self.batch_size       = config["batch_size"]
        self.num_workers      = config["num_workers"]
        self.g_lr             = config["g_lr"]
        self.d_lr             = config["d_lr"]
        self.lr_decay         = config["lr_decay"]
        self.beta1            = config["beta1"]
        self.beta2            = config["beta2"]
        self.start_train_iteration = config["start_train_iteration"]

        self.dataset           = config["dataset"]
        self.models_path       = config["models_path"]
        self.results_path      = config["results_path"]
        self.log_interval      = config["log_interval"]
        self.sampling_interval = config["sampling_interval"]
        self.checkp_interval   = config["checkp_interval"]

        self.build_model()

        # Start with trained model
        if self.start_train_iteration:
            self.load_pretrained_model()

    def train(
        self,
        results      = None,
        start_epoch  = 0,
        device       = device,
        ):
        '''
        Implements the SAGAN training loop.
        '''

        # Batch of latent (noise) vectors for evaluating /
        # visualizing the training progress of the generator

        ref_batch_size = 25
        mean_gp        = 0.0

        assert self.log_interval % self.ncritic == 0, \
            f'Log interval is {self.log_interval} and must be a multiple of {self.ncritic}'

        gen_log_interval = int(self.log_interval / self.ncritic)
        print(f'Log interval according to the generator iterations: {gen_log_interval}')

        # Fixed input batch of random vectors to used to evaluate the generation
        # quality during training
        fixed_z = torch.randn(ref_batch_size, self.z_dim, device=device)

        for epoch in range(start_epoch, start_epoch+self.epochs):

            ts  = time.time()

            for i, imgs in enumerate(self.dataloader):

                step       = epoch * len(self.dataloader) + i + 1
                batch_size = imgs.size(0)

                # real images
                real_images = imgs.to(device)

                # generated images
                z  = torch.randn(
                    batch_size,
                    self.z_dim,
                    device = device,
                )  # format BS,C,H,W

                fake_images = self.generator(z)

                # --------------------------
                # Train the discriminator
                # --------------------------

                self.discriminator.train()
                self.generator.train()

                # Compute loss with real images.
                # d_attn1r, d_attn2r, d_attn1f, d_attn2f, g_attn1, g_attn2
                # are the attention scores.

                real_images                    = real_images.to(device)
                d_out_real, d_attn1r, d_attn2r = self.discriminator(real_images)

                if self.adv_loss == 'wgan-gp':
                    d_loss_real = - torch.mean(d_out_real)
                elif self.adv_loss == 'hinge':
                    d_loss_real = torch.nn.ReLU()(1.0 - d_out_real).mean()

                # Apply Gumbel softmax

                z = torch.randn(real_images.size(0), self.z_dim, device=device)
                fake_images, g_attn1, g_attn2   = self.generator(z)
                d_out_fake, d_attn1f, d_attn2f  = self.discriminator(fake_images)

                if self.adv_loss == 'wgan-gp':
                    d_loss_fake = d_out_fake.mean()
                elif self.adv_loss == 'hinge':
                    d_loss_fake = torch.nn.ReLU()(1.0 + d_out_fake).mean()

                # Backpropgate the loss gradients and update model weights
                d_loss = d_loss_real + d_loss_fake
                self.reset_optimizer_grads()
                d_loss.backward()
                self.d_optimizer.step()

                if self.adv_loss == 'wgan-gp':

                    # Compute the gradient penalty
                    alpha        = torch.rand(real_images.size(0), 1, 1, 1).to(device)
                    alpha        = alpha.expand_as(real_images)
                    interpolated = Variable(
                        alpha * real_images.data + (1-alpha) * fake_images.data,
                        requires_grad=True,
                    )
                    out , _ , _ = self.discriminator(interpolated)

                    grad = torch.autograd.grad(
                        outputs      = out,
                        inputs       = interpolated,
                        grad_outputs = torch.ones(out.size()).to(device),
                        retain_graph = True,
                        create_graph = True,
                        only_inputs  = True,
                    )[0]

                    grad        = grad.view(grad.size(0), -1)
                    grad_l2norm = torch.sqrt(torch.sum(grad ** 2, dim=1))
                    gp          = torch.mean((grad_l2norm - 1) ** 2)

                    # Backpropgate the loss gradients and update model weights
                    d_loss_gp   = self.lambda_gp * gp

                    self.reset_optimizer_grads()
                    d_loss_gp.backward()
                    self.d_optimizer.step()

                    d_loss += d_loss_gp

                if step % self.ncritic == 0:

                    # --------------------------------
                    # Train the generator and gumbel
                    # --------------------------------

                    # Create random noise

                    z = torch.randn(real_images.size(0), self.z_dim).to(device)
                    fake_images,_,_ = self.generator(z)

                    # Compute loss with fake images

                    g_out_fake,_,_ = self.discriminator(fake_images)  # batch x n
                    g_loss         = - g_out_fake.mean()

                    self.reset_optimizer_grads()
                    g_loss.backward()
                    self.g_optimizer.step()

                    # Save the results in a dictionary ...................................

                    results["d_loss"].append(d_loss.item())
                    results["d_r_loss"].append(d_loss_real.item())
                    results["d_f_loss"].append(d_loss_fake.item())
                    results["g_loss"].append(g_loss.item())
                    results["d_gamma1"].append(self.discriminator.attn1.gamma.mean().item())
                    results["d_gamma2"].append(self.discriminator.attn2.gamma.mean().item())
                    results["g_gamma1"].append(self.generator.attn1.gamma.mean().item())
                    results["g_gamma2"].append(self.generator.attn2.gamma.mean().item())
                    if self.adv_loss == 'wgan-gp':
                        results["gp"].append(gp.item())

                # Print progress information ...........................................

                if step % self.log_interval == 0:

                    mean_d_loss      = np.mean(results["d_loss"][-gen_log_interval:])
                    mean_d_r_loss    = np.mean(results["d_r_loss"][-gen_log_interval:])
                    mean_d_f_loss    = np.mean(results["d_f_loss"][-gen_log_interval:])
                    mean_g_loss      = np.mean(results["g_loss"][-gen_log_interval:])
                    d_gamma1         = self.discriminator.attn1.gamma.mean().item()
                    d_gamma2         = self.discriminator.attn2.gamma.mean().item()
                    g_gamma1         = self.generator.attn1.gamma.mean().item()
                    g_gamma2         = self.generator.attn2.gamma.mean().item()
                    if self.adv_loss == 'wgan-gp':
                        mean_gp      = np.mean(results["gp"][-gen_log_interval:])

                    print(f'epoch|iter: {epoch+1 :4d} | {i+1 :5d} / {len(self.dataloader) :6d}', end = " ")
                    print(f'({((i+1)*100)/len(self.dataloader) :0>5.1f}%)', end=" ")
                    print(f'D_loss: {mean_d_loss :0>7.5f}', end=" ")
                    print(f'G_loss: {mean_g_loss :0>7.5f}', end=" ")
                    print(f'D_g1: {d_gamma1 :0>6.4f} D_g2: {d_gamma2 :0>6.4f}', end=" ")

                    if self.adv_loss == 'wgan-gp':
                        print(f'G_g1: {g_gamma1 :0>6.4f} G_g2: {g_gamma2 :0>6.4f}', end=" ")
                        print(f'GP: {mean_gp :0>.6f}')
                    else:
                        print(f'G_g1: {g_gamma1 :0>6.4f} G_g2: {g_gamma2 :0>6.4f}')

                    try:
                        # Log metrics to Weights and Biases .........................
                        wandb.log(
                            {
                            "d_loss":    mean_d_loss,
                            "d_r_loss":  mean_d_r_loss,
                            "d_f_loss":  mean_d_f_loss,
                            "g_loss":    mean_g_loss,
                            "gp":        mean_gp,
                            "d_gamma1":  d_gamma1,
                            "d_gamma2":  d_gamma2,
                            "g_gamma1":  g_gamma1,
                            "g_gamma2":  g_gamma2,
                            }
                        )
                    except Exception as ex:
                        print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

            # End of an epoch .....................................................

            te        = time.time()
            texec_sec = te - ts
            texec_str = time_format(texec_sec)
            print(f'Epoch training time: {texec_str}')
            results['epoch_training_time'].append(texec_sec)

            try:
                wandb.log(
                    {
                    "epoch_training_time_sec": texec_sec,
                    }
                )
            except Exception as ex:
                print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

            # Sample images .......................................................

            if ((epoch+1) % self.sampling_interval == 0) or ((epoch+1) == (start_epoch+self.epochs)):
                self.generator.eval()
                fake_images , _ , _ = self.generator(fixed_z)
                f_name      = f'results/{self.experiment_name}/generated_fixedZ_epoch{str(epoch+1).zfill(3)}.png'
                save_image(
                    denorm(fake_images),
                    f_name,
                    nrow = 5,
                )

            # Save a model checkpoint .............................................

            if ((epoch+1) % self.checkp_interval == 0) or ((epoch+1) == (start_epoch+self.epochs)):
                file_save_model = f'models/{self.experiment_name}/{self.experiment_name}_{str(epoch+1).zfill(3)}.pth'
                self.save_model_and_results(
                    results,
                    epoch+1,
                    config,
                    file_save_model,
                )


    def build_model(self):
        '''
        Instantiate the models, optimizers and loss function.
        '''

        # batch_size, image_size=64, channels=3, z_dim=128, conv_dim=64
        # Instantiate the generator model
        self.generator = Generator(
            self.batch_size,
            self.image_size,
            self.channels,
            self.z_dim,
            self.g_conv_dim,
        ).to(device)

        # Instantiate the discriminator model
        self.discriminator = Discriminator(
            self.batch_size,
            self.image_size,
            self.channels,
            self.d_conv_dim,
        ).to(device)

        if self.parallel:
            self.generator     = nn.DataParallel(self.generator)
            self.discriminator = nn.DataParallel(self.discriminator)

        # Loss and optimizer

        self.g_optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.generator.parameters()),
            lr = self.g_lr,
            betas = [self.beta1, self.beta2],
        )
        self.d_optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.discriminator.parameters()),
            lr = self.d_lr,
            betas = [self.beta1, self.beta2],
        )

        # The base loss function
        self.c_loss = torch.nn.CrossEntropyLoss()

        # Print the model's architecture

        aux_data = torch.randn(self.batch_size, self.z_dim, device=device)
        summ = summary(
            self.generator,
            input_data   = aux_data,
            col_width    = 16,
            col_names    = ["kernel_size", "output_size", "num_params"],
            row_settings = ["var_names"],
        )
        print(summ)

        aux_data = torch.randn(
            (
            self.batch_size,
            self.channels,
            self.image_size,
            self.image_size,
            )
        ).to(device)

        summ = summary(
            self.discriminator,
            input_data   = aux_data,
            col_width    = 16,
            col_names    = ["kernel_size", "output_size", "num_params"],
            row_settings = ["var_names"],
        )
        print(summ)

    def save_model_and_results(
            self,
            results,
            epoch,
            hyperparameters,
            file_name,
        ):
        '''
        Given the current Trainer object, this method saves to a file:
        (i)   the discriminator model weights,
        (ii)  the generator model weights,
        (iii) the discriminator optimizer state,
        (iv)  the generator optimizer state,
        (v)   the results saved during model training,
        (vi)  the actual training epoch number,
        (vii) the hyperparameters used to train the models.
        '''
        results_to_save = {
            'discriminator':   self.discriminator.state_dict(),
            'generator':       self.generator.state_dict(),
            'd_optimizer':     self.d_optimizer.state_dict(),
            'g_optimizer':     self.g_optimizer.state_dict(),
            'results':         results,
            'epoch':           epoch,
            'hyperparameters': hyperparameters,
        }

        torch.save(
            results_to_save,
            file_name,
        )


    def load_model(self, file_name, device):
        '''
        Given the current Trainer object, this method loads from file 'file_name':
        (i)   the discriminator model weights,
        (ii)  the generator model weights,
        (iii) the discriminator optimizer state,
        (iv)  the generator optimizer state,
        (v)   the results saved during model training,
        (vi)  the training epoch number when the checkpoint was saved,
        (vii) the hyperparameters used to train the models.
        and put the models on 'device'.

        Returns the loaded results, the loaded epoch number, and the loaded hyperparameters.
        '''
        results_loaded = torch.load(file_name)

        self.discriminator.load_state_dict(results_loaded['discriminator'])
        self.discriminator.to(device)

        self.generator.load_state_dict(results_loaded['generator'])
        self.generator.to(device)

        self.d_optimizer.load_state_dict(results_loaded['d_optimizer'])
        self.g_optimizer.load_state_dict(results_loaded['g_optimizer'])

        # Returns the saved results and the saved hyperparameters
        return results_loaded['results'], results_loaded['epoch'], results_loaded['hyperparameters']


    def reset_optimizer_grads(self):
        '''
        Reset the optimizer gradientd at the iteration/minibatch begin.
        '''
        self.d_optimizer.zero_grad()
        self.g_optimizer.zero_grad()


    def generate_grid_images(self, num_grids, grid_W_H, experiment_name, device):

        grid_size = grid_W_H ** 2
        assert self.batch_size >= grid_size, f'Grid size must be less or equal to batch size={self.batch_size}'

        self.generator.eval()

        with torch.inference_mode():

            for num in range(num_grids):

                # Generate a set of latent vectors
                noise = torch.randn(self.batch_size, self.z_dim, device=device)

                # Generate a set of fake images with generator
                img , act1 , act2 = self.generator(noise)
                img  = img.detach().cpu()
                act1 = act1.detach().cpu()
                act2 = act2.detach().cpu()

                if(self.batch_size > grid_size):
                    img = img[:grid_size]

                img = denorm(img)

                # Create a grid with the generated images
                grid = make_grid(img, padding=2, normalize=False)
                grid = grid.permute(1, 2, 0)
                grid = grid.numpy()

                # Display the grid of images
                _ = plt.figure(figsize=(10, 10), constrained_layout=True)
                plt.imshow(grid)
                
                # Save the grid of images as a PNG file
                file_png = f'results/{experiment_name}/{experiment_name}_generated_final_{str(num+1).zfill(3)}.png'
                plt.imsave(file_png, grid)
                plt.close() 

                # Plot the attention maps 1 and 2
                fig, axs = plt.subplots(nrows=grid_size, ncols=3, figsize=(5*3, 5*grid_size))
                for r in range(grid_size):
                    axs[r][0].imshow(img[r].cpu().permute(1,2,0).numpy())
                    axs[r][0].set_title(f'Generated image #{r}')
                    axs[r][1].imshow(act1[r].cpu().numpy(), cmap='gray')
                    axs[r][1].set_title('Attention map 1')
                    axs[r][2].imshow(act2[r].cpu().numpy(), cmap='gray')
                    axs[r][2].set_title('Attention map 2')
                plt.tight_layout()
                
                # Save the activation maps as a PNG file
                file_png = f'results/{experiment_name}/{experiment_name}_attention_maps_{str(num+1).zfill(3)}.png'
                plt.savefig(file_png, format='png')
                # plt.show()
                plt.close() 

## Training Loop

In [ ]:
# Create an empty dictionary to store the training results

results = {
    'd_loss':              [],
    'd_r_loss':            [],
    'd_f_loss':            [],
    'g_loss':              [],
    'd_gamma1':            [],
    'd_gamma2':            [],
    'g_gamma1':            [],
    'g_gamma2':            [],
    'gp':                  [],
    'epoch_training_time': [],
}

In [ ]:
# For fast training
cudnn.benchmark = True

# Instantiate the DataLoader

dataloader = Data_Loader(
    dataset     = config["dataset"],
    images_path = config["data_path"],
    image_size  = config["image_size"],
    crop_size   = config["crop_size"],
    batch_size  = config["batch_size"],
    shuffle     = True,
)

# Create directories if they do not exist

make_folder(config["models_path"])
make_folder(config["results_path"])

if config["model"] == 'sagan':
    trainer = Trainer(dataloader.loader(), config)

elif config["model"] == 'qgan':
    pass

In [ ]:
# =========================================================================
# Train the model from the beginning
# =========================================================================

if LOAD_TRAINED_MODEL == False and SKIP_TRAIN_MODEL == False:

    start_epoch = 0

    trainer.train(
        results      = results,
        start_epoch  = start_epoch,
        device       = device,
    )

# =========================================================================
# Load the saved models
# =========================================================================

elif LOAD_TRAINED_MODEL == True:

    file_save_model = f'models/{config["experiment_name"]}.pth'
    results, start_epoch, _ = trainer.load_model(
        file_save_model,
        device,
    )

    # ---------------------------------------------------------------------
    # Continue training of the loaded models
    # ---------------------------------------------------------------------

    if SKIP_TRAIN_MODEL == False:

        trainer.train(
            results      = results,
            start_epoch  = start_epoch,
            device       = device,
        )

## Export results to a CSV file

In [ ]:
import pandas as pd

results_df = pd.DataFrame(
    list(
        zip(
            results['d_loss'],
            results['g_loss'],
            results['gp'],
            results['d_gamma1'],
            results['d_gamma2'],
            results['g_gamma1'],
            results['g_gamma2'],
        )
    ),
    columns = ['d_loss', 'g_loss', 'gp', 'd_gamma1', 'd_gamma2', 'g_gamma1', 'g_gamma2']
)
results_df.head()

file_name = f'results/{config["experiment_name"]}_results.csv'
results_df.to_csv(file_name)

## Generate grids of images with the fully trained generator

In [ ]:
trainer.generate_grid_images(
    num_grids       = 8,
    grid_W_H        = 8,
    experiment_name = config["experiment_name"],
    device          = device,
)

In [ ]:
# Mark the Weights and Bias run as finished
wandb.finish()